In [0]:
%run ../utils

In [0]:
# Create lead_activites_processed_custom from the SILVER parent
# lead_activites_processed — the wide custom-field pivot for the ACTIVITY
# family (mirrors 2_1_02_Silver_leads_processed_custom for leads).
#
# Shape: one row per TYPED activity (custom_activity_type_id IS NOT NULL —
# custom values exist only on typed rows), one column per distinct cf_ key
# from the parent's custom_fields MAP, named custom_<cf_ key> (the
# contacts-child naming convention). Audit + PK + type id carried unchanged.
#
# CDC: parent-driven delete+append on the child watermark (same pattern as
# 2_4_02). The pivot key set is scanned from the FULL parent on every run —
# not just the CDC batch — so the column set is stable and a schema
# mismatch can never fail the append.
import time
from pyspark.sql.functions import col, lit, explode
start = time.time()

print(f"{'='*80}")
print(f"lead_activites_processed -> lead_activites_processed_custom")
print(f"{'='*80}")

PARENT_TABLE = 'crm_ingestion.silver.lead_activites_processed'
CHILD_TABLE = 'lead_activites_processed_custom'

# Check CDC watermark for the child
watermark = get_silver_watermark(CHILD_TABLE)
if watermark:
    print(f"  CDC MODE: reading parent changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Read the silver parent (children can never be newer than the parent)
df_flat = spark.table(PARENT_TABLE)
if watermark:
    df_flat = df_flat.filter(col("bronze_updated_at") > lit(watermark))

n_changed = df_flat.count()
if n_changed == 0:
    print(f"  CDC: No new parent rows since {watermark}")
    tdf = spark.table(f"crm_ingestion.silver.{CHILD_TABLE}")
    print(f"  Existing {CHILD_TABLE}: {tdf.count():,} rows, {len(tdf.columns)} cols")
else:
    print(f"  Source: {n_changed:,} changed parent rows")

    # Step 2: Typed activities only — values exist only there
    typed = df_flat.where(col("custom_activity_type_id").isNotNull())
    print(f"  Typed activities in batch: {typed.count():,}")

    # Step 3: Stable column set — full-parent key scan (not batch-local)
    keys = [r["k"] for r in
            spark.table(PARENT_TABLE)
                 .select(explode("custom_fields").alias("k", "v"))
                 .select("k").distinct().collect()]
    print(f"  Pivot: {len(keys)} distinct cf_ keys")

    # Step 4: Pivot — map lookup per key, audit + PK + type id carried
    out = (typed
           .withColumnRenamed("id", "lead_activites_processed_id")
           .select("bronze_insert_date", "bronze_updated_at",
                   "lead_activites_processed_id", "custom_activity_type_id",
                   *[col("custom_fields").getItem(k).alias(f"custom_{k}")
                     for k in keys]))

    # Step 5: Write (full overwrite on first run, delete+append on CDC)
    _CDC_WATERMARK = watermark
    write_silver(out, CHILD_TABLE)
    _CDC_WATERMARK = None

    print(f"  Columns written: {len(out.columns)}")
    print(f"\n  Completed in {time.time() - start:.1f}s")
print(f"{'='*80}")